# Residual(t) = Daily anomaly(t) − MJO(t)

## 1. Checking the input files=>

In [5]:
import xarray as xr

# =====================================================
# FILE PATHS
# =====================================================

anom_file = r"D:\PHD 2025~\MJO_Server1\anomaly.nc"
mjo_file  = r"D:\PHD 2025~\MJO_Server1\mjo_lanczos.nc"

# =====================================================
# FUNCTION TO PRINT INFO
# =====================================================

def inspect_dataset(file_path, name):

    print("\n" + "="*70)
    print(f"📂 FILE: {name}")
    print("="*70)

    ds = xr.open_dataset(file_path)

    # -------------------------
    # FULL DATASET STRUCTURE
    # -------------------------
    print("\n📊 DATASET STRUCTURE:")
    print(ds)

    # -------------------------
    # VARIABLES
    # -------------------------
    print("\n📦 VARIABLES:")
    for var in ds.data_vars:
        print(" -", var)

    # -------------------------
    # DIMENSIONS
    # -------------------------
    print("\n📐 DIMENSIONS:")
    for dim in ds.dims:
        print(f" - {dim}: {ds.dims[dim]}")

    # -------------------------
    # COORDINATES
    # -------------------------
    print("\n🌍 COORDINATES:")
    for coord in ds.coords:
        print(" -", coord)

    # -------------------------
    # MAIN VARIABLE DETAILS
    # -------------------------
    var_name = list(ds.data_vars)[0]
    da = ds[var_name]

    print("\n🔎 MAIN VARIABLE:", var_name)
    print("Shape:", da.shape)
    print("Dims :", da.dims)

    # -------------------------
    # ATTRIBUTES
    # -------------------------
    print("\n📏 ATTRIBUTES:")
    for k, v in da.attrs.items():
        print(f" - {k}: {v}")

    # -------------------------
    # SAMPLE VALUES
    # -------------------------
    print("\n🔢 SAMPLE VALUES:")
    print(da.values.flatten()[:5])

    # -------------------------
    # MIN / MAX
    # -------------------------
    print("\n📊 RANGE:")
    print("Min:", float(da.min()))
    print("Max:", float(da.max()))

    print("\n✅ Done inspecting:", name)


# =====================================================
# RUN INSPECTION
# =====================================================

inspect_dataset(anom_file, "ANOMALY FILE")
inspect_dataset(mjo_file,  "MJO (LANCZOS) FILE")


📂 FILE: ANOMALY FILE

📊 DATASET STRUCTURE:
<xarray.Dataset> Size: 4GB
Dimensions:     (valid_time: 16802, latitude: 201, longitude: 321)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 134kB 1980-01-01 ... 2025-12-31
  * latitude    (latitude) float64 2kB 50.0 49.75 49.5 49.25 ... 0.5 0.25 0.0
  * longitude   (longitude) float64 3kB 40.0 40.25 40.5 ... 119.5 119.8 120.0
Data variables:
    tp          (valid_time, latitude, longitude) float32 4GB ...
Attributes:
    CDI:                     Climate Data Interface version 2.4.0 (https://mp...
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    history:                 Wed Mar 25 17:11:49 2026: cdo ydaysub ERA5_1980_...
    CDO:                     Climate Data Operators version 2.4.0 (https://mp...

📦 VARIABLES:
 - tp

📐 DIMENSIONS:
 - valid_time: 

C:\Users\hp\AppData\Local\Temp\ipykernel_3824\2810611059.py:40: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f" - {dim}: {ds.dims[dim]}")


[ 3.4736549e-05  8.6245345e-06 -9.1531983e-06 -2.2940010e-05
 -2.9284020e-05]

📊 RANGE:
Min: -0.004199650138616562
Max: 0.05376053228974342

✅ Done inspecting: ANOMALY FILE

📂 FILE: MJO (LANCZOS) FILE

📊 DATASET STRUCTURE:
<xarray.Dataset> Size: 9GB
Dimensions:    (latitude: 201, longitude: 321, time: 16802)
Coordinates:
  * latitude   (latitude) float64 2kB 0.0 0.25 0.5 0.75 ... 49.5 49.75 50.0
  * longitude  (longitude) float64 3kB 40.0 40.25 40.5 ... 119.5 119.8 120.0
  * time       (time) datetime64[ns] 134kB 1980-01-01 1980-01-02 ... 2025-12-31
Data variables:
    tp_mjo     (latitude, longitude, time) float64 9GB ...

📦 VARIABLES:
 - tp_mjo

📐 DIMENSIONS:
 - latitude: 201
 - longitude: 321
 - time: 16802

🌍 COORDINATES:
 - time
 - longitude
 - latitude

🔎 MAIN VARIABLE: tp_mjo
Shape: (201, 321, 16802)
Dims : ('latitude', 'longitude', 'time')

📏 ATTRIBUTES:
 - standard_name: unknown
 - long_name: Total precipitation
 - units: m
 - GRIB_paramId: 228
 - GRIB_dataType: fc
 - GRIB_num

C:\Users\hp\AppData\Local\Temp\ipykernel_3824\2810611059.py:40: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f" - {dim}: {ds.dims[dim]}")


[-0.00936958 -0.01165906 -0.01379285 -0.01570834 -0.01734664]

📊 RANGE:
Min: -2.657104689763329
Max: 3.9521874865505517

✅ Done inspecting: MJO (LANCZOS) FILE


# 2. Experimental residual file generation=>

In [2]:
import xarray as xr
import numpy as np

# =====================================================
# LOAD DATA
# =====================================================

ds_anom = xr.open_dataset(r"D:\PHD 2025~\MJO_Server1\anomaly.nc")
ds_mjo  = xr.open_dataset(r"D:\PHD 2025~\MJO_Server1\mjo_lanczos.nc")

#anom_file = r"D:\PHD 2025~\MJO_Server1\anomaly.nc"
#mjo_file  = r"D:\PHD 2025~\MJO_Server1\mjo_lanczos.nc"


# Fix dimension name
ds_anom = ds_anom.rename({"valid_time": "time"})

# Get variables
var_anom = list(ds_anom.data_vars)[0]
var_mjo  = list(ds_mjo.data_vars)[0]

anom = ds_anom[var_anom]
mjo  = ds_mjo[var_mjo]

# =====================================================
# FIX DIMENSION ORDER (CRITICAL)
# =====================================================

mjo = mjo.transpose("time", "latitude", "longitude")

# =====================================================
# SELECT OCTOBER 2025
# =====================================================

anom_oct = anom.sel(time="2025-10")
mjo_oct  = mjo.sel(time="2025-10")

# =====================================================
# ALIGN DATA
# =====================================================

anom_oct, mjo_oct = xr.align(anom_oct, mjo_oct)

# =====================================================
# CONVERT UNITS (m → mm)
# =====================================================

anom_oct = anom_oct * 1000
mjo_oct  = mjo_oct * 1000

# =====================================================
# CLEAN DATA
# =====================================================

anom_oct = anom_oct.where(np.isfinite(anom_oct))
mjo_oct  = mjo_oct.where(np.isfinite(mjo_oct))

# 🔥 REMOVE EXTREME MJO VALUES (IMPORTANT)
mjo_oct = mjo_oct.where(np.abs(mjo_oct) < 100)

# =====================================================
# COMPUTE RESIDUAL
# =====================================================

residual = anom_oct - mjo_oct

# =====================================================
# METADATA
# =====================================================

residual.name = "non_mjo_precip"
residual.attrs["units"] = "mm/day"
residual.attrs["description"] = "Residual rainfall = anomaly - MJO (Lanczos)"

# =====================================================
# SAVE
# =====================================================

output_file = r"D:\PHD 2025~\PHD\8.Residual_Prep\non_mjo_oct2025.nc"
residual.to_netcdf(output_file)

print("\n💾 Saved:", output_file)

# =====================================================
# CHECK
# =====================================================

print("\n📊 Residual stats:")
print("Min:", float(residual.min()))
print("Max:", float(residual.max()))

print("\n✅ SUCCESS — residual computed correctly")


💾 Saved: D:\PHD 2025~\PHD\8.Residual_Prep\non_mjo_oct2025.nc

📊 Residual stats:
Min: -100.57224735931999
Max: 109.53015082462885

✅ SUCCESS — residual computed correctly
